# 한국어 로컬 LLM 비교 — 회의록 근거 답변 생성

`main.py` 채팅(`/api/v1/chat`)의 **답변 생성 단계**에 쓸 로컬 LLM을 고르기 위한 비교 노트북입니다.
프로젝트의 `.py` 파일을 import하지 않고 모든 코드를 셀 안에 담았습니다.

**비교 방식**
1. 질문 8개(대표 이슈 7개 + 근거에 없는 내용을 묻는 미끼 질문 1개)마다 PostgreSQL에서 회의록 근거 6건을 **한 번만** 검색해 저장합니다.
   모든 모델이 **같은 근거·같은 프롬프트**를 받으므로 차이는 모델에서만 생깁니다.
2. 프롬프트는 `api/services.py` `RadarApiService.chat()`의 시스템 프롬프트와 근거 블록 형식을 그대로 씁니다.
3. 모델마다 워밍업 후 8개 질문에 답하게 하고, 끝나면 모델을 메모리에서 내립니다(다음 모델이 GPU를 온전히 쓰도록).
4. 답변을 자동 지표로 채점합니다: 응답 시간·생성 속도, 한글 비율, `[근거 N]` 인용, **근거에 없는 숫자·날짜·의회명**(검증),
   미끼 질문에서 "근거 부족"이라고 답했는지, `<think>` 노출 여부.

**사전 조건**: pgvector DB(`council_arctic_raw_v1`), 로컬 Ollama와 비교할 모델들, HuggingFace 캐시의 임베딩 모델.
설정은 같은 폴더의 `.env`를 읽고, 없으면 제출 폴더 루트의 `../.env`를 씁니다.
결과는 제출 폴더의 `data/llm_compare/`에 저장되며, 다시 실행하면 이미 끝난 (모델, 질문)은 건너뜁니다.

> 자동 지표는 사람 평가를 대신하지 않습니다. 최종 선택 전에 6번 셀에서 답변 원문을 직접 읽어 보세요.

## 1. 설정

In [1]:
from __future__ import annotations

import json
import os
import re
import shlex
import statistics
import time
import urllib.request
from collections import defaultdict
from datetime import date, datetime
from pathlib import Path
from typing import Any
from urllib.parse import urlparse

from IPython.display import Markdown, display

BASE_DIR = Path.cwd()          # 이 노트북이 있는 폴더에서 실행한다
OUT_DIR = BASE_DIR.parent / "data" / "llm_compare"   # 제출 폴더의 data/llm_compare
OUT_DIR.mkdir(parents=True, exist_ok=True)


def load_env_file(path: Path) -> int:
    """KEY=VALUE 형식의 .env를 읽는다. 셸에서 이미 지정한 값은 덮어쓰지 않는다."""
    if not path.is_file():
        return 0
    loaded = 0
    for raw in path.read_text(encoding="utf-8-sig").splitlines():
        line = raw.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, value = line.removeprefix("export ").split("=", 1)
        key = key.strip()
        if key in os.environ:
            continue
        try:
            value = " ".join(shlex.split(value, comments=True, posix=True))
        except ValueError:
            value = value.strip().strip("\"'")
        os.environ[key] = value
        loaded += 1
    return loaded


for env_path in (BASE_DIR / ".env", BASE_DIR.parent / ".env"):
    if env_path.is_file():
        print(f"{env_path} 에서 {load_env_file(env_path)}개 설정 로드")
        break
else:
    print(".env를 찾지 못했습니다. 환경 변수만 사용합니다.")

SETTINGS = {
    "dsn": os.environ.get("PGVECTOR_DSN", ""),
    "table": os.environ.get("COUNCIL_PG_TABLE", "council_arctic_raw_v1"),
    "ef_search": max(1, int(os.environ.get("COUNCIL_PG_EF_SEARCH", "200"))),
    "embedding_model": os.environ.get("COUNCIL_EMBEDDING_MODEL", "dragonkue/snowflake-arctic-embed-l-v2.0-ko"),
    "embedding_revision": os.environ.get("COUNCIL_EMBEDDING_REVISION", "55ec6e9358a56d56af759bc8372e970caf8c305f"),
    "max_seq_length": int(os.environ.get("COUNCIL_MAX_SEQ_LENGTH", "2048")),
    "device": os.environ.get("COUNCIL_DEVICE", "auto"),
    "model_cache": os.environ.get("COUNCIL_MODEL_CACHE"),
    "allow_download": os.environ.get("COUNCIL_ALLOW_DOWNLOAD", "0") == "1",
    "ollama_url": os.environ.get("OLLAMA_URL", "http://127.0.0.1:11434"),
}
if not SETTINGS["dsn"]:
    raise RuntimeError("PGVECTOR_DSN이 없습니다 (.env 확인)")

# 비교할 모델 (Ollama 이름). 첫 줄이 기존 main.py 기본 모델이다.
MODELS = [
    "exaone3.5:7.8b",                                                          # LG AI연구원 (기존 기본값)
    "hf.co/mykor/A.X-4.0-Light-gguf:Q4_K_M",                                   # SK텔레콤 A.X 4.0 Light
    "hf.co/mradermacher/kanana-1.5-8b-instruct-2505-GGUF:Q4_K_M",              # 카카오 Kanana 1.5
    "hf.co/DevQuasar/trillionlabs.Tri-7B-GGUF:Q4_K_M",                         # 트릴리온랩스 Tri-7B
    "hf.co/rippertnt/HyperCLOVAX-SEED-Text-Instruct-1.5B-Q4_K_M-GGUF:Q4_K_M",  # 네이버 HyperCLOVA X SEED 1.5B
    "hf.co/LGAI-EXAONE/EXAONE-4.0-1.2B-GGUF:Q4_K_M",                           # LG EXAONE 4.0 1.2B
]
TOP_K = 6              # 질문당 근거 수 (ChatRequest 기본값)
EVIDENCE_CHARS = 1800  # 근거 1건당 프롬프트에 넣는 글자 수 (services.py와 동일)
TIMEOUT = 600          # 모델 응답 대기 시간(초)
# Ollama 기본 컨텍스트(4096 토큰)는 근거 6건 프롬프트(약 4,100토큰)보다 작아 앞부분이 잘릴 수 있다.
# 모든 모델에 같은 값을 주어 잘림 없이 비교한다. None이면 Ollama 기본값을 쓴다.
NUM_CTX: int | None = 8192
print(f"모델 {len(MODELS)}개 · 근거 {TOP_K}건 · num_ctx {NUM_CTX} · 결과 폴더 {OUT_DIR}")

..\.env 에서 15개 설정 로드
모델 6개 · 근거 6건 · num_ctx 8192 · 결과 폴더 ..\data\llm_compare


## 2. 비교 질문
대표 이슈 7개에서 하나씩 뽑은 실제 정책 질문과, 회의록에 있을 수 없는 내용을 묻는 **미끼 질문**(`hallucination_bait`)입니다.
미끼 질문에는 "근거가 부족하다"고 답하는 모델이 좋은 모델입니다.

In [2]:
QUESTIONS = [
    {"id": "birth_rate", "q": "출산율 저하에 대응하기 위해 지방의회에서 어떤 정책이 논의됐나요?"},
    {"id": "population_decline", "q": "지방소멸과 인구감소 대응으로 의회에서 제안된 사업을 알려줘"},
    {"id": "youth_jobs", "q": "청년 일자리와 지역정착 지원에 대해 의원들이 지적한 문제점은?"},
    {"id": "jeonse_fraud", "q": "전세사기 피해자 지원 대책으로 어떤 내용이 나왔어?"},
    {"id": "regional_healthcare", "q": "지역 필수의료 공백 문제에 대해 회의록에서 어떤 대안이 언급됐나요?"},
    {"id": "school_violence", "q": "학교폭력 피해 학생 보호를 위해 논의된 내용은?"},
    {"id": "flood_safety", "q": "집중호우 침수 피해 예방 대책으로 어떤 사업 예산이 논의됐는지 금액과 함께 알려줘"},
    {"id": "hallucination_bait", "q": "2025년 화성 이주민 지원 조례를 처음 통과시킨 의회와 예산 규모는?"},
]
for item in QUESTIONS:
    print(f"{item['id']:20} {item['q']}")

birth_rate           출산율 저하에 대응하기 위해 지방의회에서 어떤 정책이 논의됐나요?
population_decline   지방소멸과 인구감소 대응으로 의회에서 제안된 사업을 알려줘
youth_jobs           청년 일자리와 지역정착 지원에 대해 의원들이 지적한 문제점은?
jeonse_fraud         전세사기 피해자 지원 대책으로 어떤 내용이 나왔어?
regional_healthcare  지역 필수의료 공백 문제에 대해 회의록에서 어떤 대안이 언급됐나요?
school_violence      학교폭력 피해 학생 보호를 위해 논의된 내용은?
flood_safety         집중호우 침수 피해 예방 대책으로 어떤 사업 예산이 논의됐는지 금액과 함께 알려줘
hallucination_bait   2025년 화성 이주민 지원 조례를 처음 통과시킨 의회와 예산 규모는?


## 3. Ollama 확인
회의록이 외부로 나가지 않도록 localhost Ollama만 허용합니다. 설치되지 않은 모델은 비교에서 빼고 받는 명령을 알려 줍니다.

In [3]:
LOCAL_HOSTS = {"localhost", "127.0.0.1", "::1"}
OLLAMA = SETTINGS["ollama_url"].rstrip("/")
if (urlparse(OLLAMA).hostname or "").lower() not in LOCAL_HOSTS:
    raise RuntimeError("보안 정책상 localhost Ollama만 사용할 수 있습니다")


def ollama_post(path: str, body: dict[str, Any], timeout: int = TIMEOUT) -> dict[str, Any]:
    request = urllib.request.Request(OLLAMA + path, data=json.dumps(body, ensure_ascii=False).encode("utf-8"),
                                     headers={"Content-Type": "application/json"}, method="POST")
    with urllib.request.urlopen(request, timeout=timeout) as response:
        return json.loads(response.read().decode("utf-8"))


with urllib.request.urlopen(OLLAMA + "/api/tags", timeout=5) as response:
    installed = {m["name"] for m in json.loads(response.read().decode("utf-8")).get("models") or []}
missing_models = [m for m in MODELS if m not in installed]
MODELS_TO_RUN = [m for m in MODELS if m in installed]
print(f"Ollama 연결됨 · 비교할 모델 {len(MODELS_TO_RUN)}/{len(MODELS)}개")
for model in missing_models:
    print(f"  설치 안 됨 → 건너뜀:  ollama pull {model}")

Ollama 연결됨 · 비교할 모델 6/6개


## 4. 질문별 회의록 근거 (모든 모델 공통)
질문을 임베딩해 pgvector에서 가장 가까운 청크 `TOP_K`개를 가져와 `data/llm_compare/evidence_top6.json`에 저장합니다.
LLM 비교에는 모델 간 **같은 근거**면 충분하므로 dense(HNSW) 검색만 씁니다.
(전문검색까지 섞는 하이브리드 검색은 인덱스가 없어 질문당 수 분이 걸리기 때문입니다. DB는 읽기만 합니다.)

In [4]:
import psycopg
import torch
from pgvector.psycopg import register_vector
from psycopg.rows import dict_row
from sentence_transformers import SentenceTransformer

TABLE = SETTINGS["table"]
EVIDENCE_PATH = OUT_DIR / f"evidence_top{TOP_K}.json"
evidence: dict[str, list[dict[str, Any]]] = (
    json.loads(EVIDENCE_PATH.read_text(encoding="utf-8")) if EVIDENCE_PATH.is_file() else {})
need = [item for item in QUESTIONS if item["id"] not in evidence]

if need:
    device = SETTINGS["device"]
    if device == "auto":
        device = "cuda" if torch.cuda.is_available() else "cpu"
    kwargs: dict[str, Any] = {"device": device, "local_files_only": not SETTINGS["allow_download"],
                              "revision": SETTINGS["embedding_revision"]}
    if SETTINGS["model_cache"]:
        kwargs["cache_folder"] = str(Path(SETTINGS["model_cache"]).expanduser().resolve())
    encoder = SentenceTransformer(SETTINGS["embedding_model"], **kwargs)
    encoder.max_seq_length = SETTINGS["max_seq_length"]
    prompt = {"prompt_name": "query"} if "query" in getattr(encoder, "prompts", {}) else {}
    with psycopg.connect(SETTINGS["dsn"], row_factory=dict_row) as conn:
        register_vector(conn)
        conn.execute("SELECT set_config('hnsw.ef_search', %s, false)", (str(SETTINGS["ef_search"]),))
        for item in need:
            vector = encoder.encode([item["q"]], normalize_embeddings=True, **prompt)[0].astype("float32").tolist()
            rows = conn.execute(f"""
                SELECT chunk_id, council, meeting_date, meeting_name, document,
                       1 - (embedding <=> %s::vector) AS similarity
                FROM {TABLE} WHERE embedding IS NOT NULL
                ORDER BY embedding <=> %s::vector, chunk_id LIMIT %s""", [vector, vector, TOP_K]).fetchall()
            evidence[item["id"]] = [{
                "chunk_id": str(r["chunk_id"]),
                "metadata": {"council": r["council"], "meeting_name": r["meeting_name"],
                             "meeting_date": r["meeting_date"].isoformat() if r["meeting_date"] else None},
                "similarity": round(float(r["similarity"]), 4),
                "text": r["document"] or "",
            } for r in rows]
            EVIDENCE_PATH.write_text(json.dumps(evidence, ensure_ascii=False, indent=1), encoding="utf-8")
            print(f"근거 검색 {item['id']}: {len(rows)}건")
else:
    print(f"저장된 근거 사용: {EVIDENCE_PATH}")

for item in QUESTIONS:
    rows = evidence[item["id"]]
    print(f"{item['id']:20} " + " | ".join(f"{r['metadata']['council']} {r['metadata']['meeting_date']} ({r['similarity']:.2f})"
                                           for r in rows[:3]) + " …")

<conda env>\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Loading weights:   2%|▏         | 6/391 [00:00<00:06, 55.47it/s]

Loading weights:   3%|▎         | 12/391 [00:00<00:08, 46.84it/s]

Loading weights:   4%|▍         | 17/391 [00:00<00:08, 43.90it/s]

Loading weights:   6%|▌         | 22/391 [00:00<00:08, 42.44it/s]

... (10 outputs left out)

근거 검색 regional_healthcare: 6건


근거 검색 school_violence: 6건


근거 검색 flood_safety: 6건


근거 검색 hallucination_bait: 6건
birth_rate           강원특별자치도의회 2024-11-07 (0.60) | 강원특별자치도의회 2024-11-20 (0.60) | 경상남도의회 2025-06-09 (0.60) …
population_decline   전라남도의회 2024-05-23 (0.64) | 전라남도의회 2024-05-23 (0.64) | 전북특별자치도의회 2024-06-05 (0.64) …
youth_jobs           경상북도의회 2025-03-11 (0.61) | 경상북도의회 2025-03-11 (0.56) | 전라남도의회 2025-09-09 (0.56) …
jeonse_fraud         전라남도의회 2025-03-11 (0.64) | 전라남도의회 2025-03-11 (0.64) | 전라남도의회 2025-03-11 (0.64) …
regional_healthcare  전북특별자치도의회 2024-04-19 (0.61) | 전북특별자치도의회 2024-04-19 (0.60) | 충청북도의회 2025-10-21 (0.59) …
school_violence      경기도의회 2025-07-21 (0.58) | 경기도의회 2025-07-21 (0.57) | 경기도의회 2025-07-21 (0.57) …
flood_safety         경상남도의회 2025-12-01 (0.57) | 인천광역시의회 2025-06-17 (0.56) | 인천광역시의회 2025-06-17 (0.56) …
hallucination_bait   경기도의회 2024-12-27 (0.49) | 울산광역시의회 2025-06-11 (0.48) | 울산광역시의회 2025-06-16 (0.48) …


## 5. 프롬프트·채점 함수
- **프롬프트**: `api/services.py`의 시스템 프롬프트와 `[근거 N] 의회 | 날짜` 블록 형식 그대로.
- **검증**(`verify_answer`, 원본 `agent/query_pipeline.py`와 같은 규칙): 답변 속 숫자·날짜·의회명이 근거와 질문에 있는지 대조합니다.
  연도(2000~2100)와 10 이하 정수는 제외하고, 반올림·백분율 표현은 허용합니다. 범위를 벗어난 `[근거 N]` 번호도 실패로 봅니다.

In [5]:
SYSTEM_PROMPT = (  # api/services.py RadarApiService.chat()과 동일
    "지방의회 회의록 근거를 사용해 한국어로 답하세요. 근거에 없는 사실이나 숫자는 만들지 말고, "
    "각 사실 뒤에 [근거 N]을 표시하세요. 근거가 부족하면 그렇게 말하세요."
)


def build_messages(question: str, rows: list[dict[str, Any]]) -> tuple[list[dict[str, str]], str]:
    blocks = [f"[근거 {i}] {r['metadata'].get('council', '미상')} | {r['metadata'].get('meeting_date', '날짜 미상')}\n"
              f"{r['text'][:EVIDENCE_CHARS]}" for i, r in enumerate(rows, start=1)]
    evidence_text = "\n\n".join(blocks)
    return [{"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": f"질문: {question}\n\nPostgreSQL 검색 근거:\n" + evidence_text}], evidence_text


# --- 의회 이름 표기 (원본 external_policy_sources.REGION, regional_stats.SHORT)
REGION = {
    "서울특별시의회": ["서울특별시"], "부산광역시의회": ["부산광역시"], "대구광역시의회": ["대구광역시"],
    "인천광역시의회": ["인천광역시"], "광주광역시의회": ["광주광역시"], "대전광역시의회": ["대전광역시"],
    "울산광역시의회": ["울산광역시"], "세종특별자치시의회": ["세종특별자치시"], "경기도의회": ["경기도"],
    "강원특별자치도의회": ["강원특별자치도", "강원도"], "충청북도의회": ["충청북도"], "충청남도의회": ["충청남도"],
    "전북특별자치도의회": ["전북특별자치도", "전라북도"], "전라남도의회": ["전라남도"], "경상북도의회": ["경상북도"],
    "경상남도의회": ["경상남도"], "제주특별자치도의회": ["제주특별자치도"],
}
SHORT_REGION = {"서울": "서울특별시의회", "부산": "부산광역시의회", "대구": "대구광역시의회", "인천": "인천광역시의회",
                "광주": "광주광역시의회", "대전": "대전광역시의회", "울산": "울산광역시의회", "세종": "세종특별자치시의회",
                "경기": "경기도의회", "강원": "강원특별자치도의회", "충북": "충청북도의회", "충남": "충청남도의회",
                "전북": "전북특별자치도의회", "전남": "전라남도의회", "경북": "경상북도의회", "경남": "경상남도의회",
                "제주": "제주특별자치도의회"}
REGION_FORMS: dict[str, str] = {}
for council, forms in REGION.items():
    REGION_FORMS[council] = council
    REGION_FORMS[council.removesuffix("의회")] = council
    for form in forms:
        REGION_FORMS[form] = council
for short, council in SHORT_REGION.items():
    REGION_FORMS[short] = council
    REGION_FORMS[short + ("도" if council.endswith("도의회") else "시")] = council   # 경북도, 부산시 …
REGION_FORMS.update({"전라북도": "전북특별자치도의회", "강원도": "강원특별자치도의회", "제주도": "제주특별자치도의회"})

NUMBER_PATTERN = re.compile(r"\d+(?:,\d{3})*(?:\.\d+)?")
KOREAN_DATE = re.compile(r"(\d{4})\s*년\s*(\d{1,2})\s*월(?:\s*(\d{1,2})\s*일)?")


def _numbers(text: str) -> list[float]:
    return [float(value.replace(",", "")) for value in NUMBER_PATTERN.findall(text)]


def _number_supported(value: float, allowed: list[float]) -> bool:
    """반올림(0.531→0.53)·백분율(0.531→53.1%) 표현까지 허용"""
    for known in allowed:
        for candidate in (known, known * 100):
            if abs(candidate - value) <= max(0.006, abs(candidate) * 0.01):
                return True
    return False


def _iso_dates(text: str) -> str:
    """'2023년 5월 25일' → '2023-05-25' (날짜 대조를 위해)"""
    return KOREAN_DATE.sub(lambda m: f"{m[1]}-{int(m[2]):02d}" + (f"-{int(m[3]):02d}" if m[3] else ""), text)


def verify_answer(answer: str, evidence_text: str, question: str, evidence_count: int) -> dict[str, Any]:
    """답변 속 숫자·날짜·의회명·근거 번호가 근거·질문에 있는지 대조 (연도·10 이하 정수는 제외)"""
    allowed = evidence_text + " " + _iso_dates(question)
    allowed_numbers = _numbers(allowed)
    bad_citations = sorted({f"[근거 {n}]" for n in map(int, re.findall(r"\[근거\s*(\d+)\]", answer))
                            if not 1 <= n <= evidence_count})
    cleaned = re.sub(r"\[(근거|외부)[^\]]*\]", " ", _iso_dates(answer))
    cleaned_for_numbers = re.sub(r"\d{4}-\d{2}(-\d{2})?", " ", cleaned)
    unknown_numbers = sorted({f"{v:g}" for v in _numbers(cleaned_for_numbers)
                              if not (v.is_integer() and (v <= 10 or 2000 <= v <= 2100))
                              and not _number_supported(v, allowed_numbers)})
    mentioned = {REGION_FORMS[f] for f in REGION_FORMS if len(f) >= 3 and f in cleaned}
    allowed_councils = {REGION_FORMS[f] for f in REGION_FORMS if len(f) >= 3 and f in allowed}
    date_pattern = re.compile(r"\d{4}-\d{2}(?:-\d{2})?")
    unknown_dates = sorted(set(date_pattern.findall(cleaned)) - set(date_pattern.findall(allowed)))
    unknown_councils = sorted(mentioned - allowed_councils)
    return {"passed": not unknown_numbers and not unknown_councils and not unknown_dates and not bad_citations,
            "unsupported": unknown_numbers[:10] + unknown_dates[:5] + bad_citations + unknown_councils}


INSUFFICIENT = re.compile(r"근거[가 ]*(부족|없)|찾을 수 없|확인되지 않|언급되지 않|포함되어 있지 않")


def score(answer: str, question: str, evidence_text: str, evidence_count: int) -> dict[str, Any]:
    hangul = len(re.findall(r"[가-힣]", answer))
    letters = len(re.findall(r"[A-Za-z가-힣一-鿿]", answer)) or 1
    check = verify_answer(answer, evidence_text, question, evidence_count)
    return {
        "chars": len(answer),
        "hangul_ratio": round(hangul / letters, 3),
        "citation_count": len(re.findall(r"\[근거\s*\d+\]", answer)),
        "distinct_citations": len({int(n) for n in re.findall(r"\[근거\s*(\d+)\]", answer)}),
        "says_insufficient": bool(INSUFFICIENT.search(answer)),
        "think_leak": "<think>" in answer or "</think>" in answer,
        "verify_passed": check["passed"],
        "unsupported": check["unsupported"],
    }


# 동작 확인: 근거에 없는 금액·의회가 들어간 답은 실패해야 한다
demo = verify_answer("부산광역시의회에서 37억 원을 편성했습니다 [근거 1]", "[근거 1] 서울특별시의회 | 2024-05-01\n예산 12억 원", "질문", 1)
print("검증 예시:", demo)

검증 예시: {'passed': False, 'unsupported': ['37', '부산광역시의회']}


## 6. 모델별 답변 생성
모델마다 ① 짧은 워밍업(모델 로딩 시간이 응답 시간에 섞이지 않게) → ② 질문 8개 답변 → ③ 메모리에서 내리기 순서로 실행합니다.
결과는 한 줄씩 `data/llm_compare/results.jsonl`에 저장되어, 중간에 멈춰도 다시 실행하면 이어서 진행합니다.
모델 6개 기준 GPU에서 10~20분 정도 걸립니다.

In [6]:
RESULTS_PATH = OUT_DIR / "results.jsonl"
OPTIONS = {"num_ctx": NUM_CTX} if NUM_CTX else {}


def load_results() -> list[dict[str, Any]]:
    if not RESULTS_PATH.is_file():
        return []
    return [json.loads(line) for line in RESULTS_PATH.read_text(encoding="utf-8").splitlines() if line.strip()]


done = {(r["model"], r["question_id"]) for r in load_results()}
with RESULTS_PATH.open("a", encoding="utf-8") as out:
    for model in MODELS_TO_RUN:
        todo = [item for item in QUESTIONS if (model, item["id"]) not in done]
        if not todo:
            print(f"[{model}] 이미 완료")
            continue
        try:
            ollama_post("/api/chat", {"model": model, "stream": False, "options": OPTIONS,
                                      "messages": [{"role": "user", "content": "안녕"}]})
        except Exception as error:
            print(f"[{model}] 워밍업 실패 → 건너뜀: {error}")
            continue
        for item in todo:
            rows = evidence[item["id"]]
            messages, evidence_text = build_messages(item["q"], rows)
            started = time.perf_counter()
            try:
                payload, error = ollama_post("/api/chat", {"model": model, "stream": False,
                                                           "messages": messages, "options": OPTIONS}), None
            except Exception as exc:
                payload, error = {}, f"{type(exc).__name__}: {exc}"
            wall = time.perf_counter() - started
            answer = str((payload.get("message") or {}).get("content") or "").strip()
            eval_count = payload.get("eval_count") or 0
            eval_seconds = (payload.get("eval_duration") or 0) / 1e9
            record = {
                "model": model, "question_id": item["id"], "question": item["q"],
                "wall_seconds": round(wall, 2), "prompt_tokens": payload.get("prompt_eval_count"),
                "output_tokens": eval_count,
                "tokens_per_second": round(eval_count / eval_seconds, 1) if eval_seconds else None,
                "error": error, "answer": answer, **score(answer, item["q"], evidence_text, len(rows)),
            }
            out.write(json.dumps(record, ensure_ascii=False) + "\n")
            out.flush()
            print(f"[{model}] {item['id']}: {wall:.1f}s · 인용 {record['citation_count']} · "
                  + ("검증 통과" if record["verify_passed"] else f"검증 실패 {record['unsupported']}")
                  + (f" · 오류 {error}" if error else ""))
        try:   # 다음 모델이 GPU 메모리를 온전히 쓰도록 내린다
            ollama_post("/api/generate", {"model": model, "keep_alive": 0}, timeout=60)
        except Exception:
            pass
print("완료:", len(load_results()), "건")

[exaone3.5:7.8b] birth_rate: 9.5s · 인용 5 · 검증 통과


[exaone3.5:7.8b] population_decline: 10.4s · 인용 5 · 검증 통과


[exaone3.5:7.8b] youth_jobs: 11.5s · 인용 3 · 검증 통과


[exaone3.5:7.8b] jeonse_fraud: 7.4s · 인용 3 · 검증 통과


[exaone3.5:7.8b] regional_healthcare: 9.9s · 인용 6 · 검증 통과


[exaone3.5:7.8b] school_violence: 8.3s · 인용 3 · 검증 통과


... (38 outputs left out)

[hf.co/LGAI-EXAONE/EXAONE-4.0-1.2B-GGUF:Q4_K_M] regional_healthcare: 3.6s · 인용 0 · 검증 통과


[hf.co/LGAI-EXAONE/EXAONE-4.0-1.2B-GGUF:Q4_K_M] school_violence: 3.4s · 인용 0 · 검증 통과


[hf.co/LGAI-EXAONE/EXAONE-4.0-1.2B-GGUF:Q4_K_M] flood_safety: 6.7s · 인용 0 · 검증 실패 ['1.2675', '1267.5', '164.7', '171', '318', '388', '388.84', '63.6', '760', '8.4519e+06']


[hf.co/LGAI-EXAONE/EXAONE-4.0-1.2B-GGUF:Q4_K_M] hallucination_bait: 4.9s · 인용 0 · 검증 실패 ['17000', '170000', '22.37', '34.7388', '347', '347.398', '3473', '46', '530', '560', '2024-12']
완료: 48 건


## 7. 결과 요약
| 지표 | 의미 | 좋은 방향 |
|---|---|---|
| 평균 응답(s) | 질문 하나에 답하는 데 걸린 시간 (워밍업 제외) | 짧을수록 |
| 생성 속도(tok/s) | 초당 생성 토큰 | 클수록 |
| 한글 비율 | 답변 글자 중 한글 비율 (영어·한자 섞임 확인) | 1.0에 가까울수록 |
| 인용 포함 문항 · 평균 인용 수 | `[근거 N]` 표시 여부·개수 | 많을수록 |
| 검증 통과 | 근거에 없는 숫자·날짜·의회명이 없는 답변 수 | 많을수록 |
| 미끼 질문 '근거 부족' | 회의록에 없는 질문에 지어내지 않고 근거 부족이라 답했는지 | 예 |

In [7]:
results = load_results()
by_model: dict[str, list[dict[str, Any]]] = defaultdict(list)
for row in results:
    by_model[row["model"]].append(row)

lines = ["| 모델 | 문항 | 오류 | 평균 응답(s) | 생성 속도(tok/s) | 평균 길이(자) | 한글 비율 | 인용 포함 문항 | 평균 인용 수 | 검증 통과 | 미끼 질문 '근거 부족' | think 노출 |",
         "|---|---|---|---|---|---|---|---|---|---|---|---|"]
for model in [m for m in MODELS if m in by_model]:
    items = by_model[model]
    ok = [r for r in items if not r["error"] and r["answer"]]
    bait = [r for r in items if r["question_id"] == "hallucination_bait"]

    def avg(key: str) -> float:
        values = [r[key] for r in ok if r[key] is not None]
        return statistics.mean(values) if values else 0.0

    lines.append(
        f"| {model} | {len(items)} | {len(items) - len(ok)} | {avg('wall_seconds'):.1f} | {avg('tokens_per_second'):.1f} | "
        f"{avg('chars'):.0f} | {avg('hangul_ratio'):.2f} | {sum(r['citation_count'] > 0 for r in ok)}/{len(ok)} | "
        f"{avg('citation_count'):.1f} | {sum(r['verify_passed'] for r in ok)}/{len(ok)} | "
        f"{'예' if bait and bait[0]['says_insufficient'] else '아니오'} | {sum(r['think_leak'] for r in ok)} |")
REPORT = "\n".join(lines)
(OUT_DIR / "report.md").write_text(REPORT + "\n", encoding="utf-8")
display(Markdown(REPORT))

| 모델 | 문항 | 오류 | 평균 응답(s) | 생성 속도(tok/s) | 평균 길이(자) | 한글 비율 | 인용 포함 문항 | 평균 인용 수 | 검증 통과 | 미끼 질문 '근거 부족' | think 노출 |
|---|---|---|---|---|---|---|---|---|---|---|---|
| exaone3.5:7.8b | 8 | 0 | 8.6 | 46.7 | 663 | 1.00 | 7/8 | 3.5 | 8/8 | 예 | 0 |
| hf.co/mykor/A.X-4.0-Light-gguf:Q4_K_M | 8 | 0 | 7.7 | 51.5 | 850 | 1.00 | 8/8 | 6.4 | 7/8 | 예 | 0 |
| hf.co/mradermacher/kanana-1.5-8b-instruct-2505-GGUF:Q4_K_M | 8 | 0 | 16.0 | 44.9 | 1044 | 0.99 | 3/8 | 1.2 | 7/8 | 예 | 0 |
| hf.co/DevQuasar/trillionlabs.Tri-7B-GGUF:Q4_K_M | 8 | 0 | 40.2 | 18.7 | 1665 | 1.00 | 4/8 | 5.6 | 6/8 | 예 | 0 |
| hf.co/rippertnt/HyperCLOVAX-SEED-Text-Instruct-1.5B-Q4_K_M-GGUF:Q4_K_M | 8 | 0 | 4.2 | 155.6 | 1168 | 1.00 | 7/8 | 6.1 | 8/8 | 아니오 | 0 |
| hf.co/LGAI-EXAONE/EXAONE-4.0-1.2B-GGUF:Q4_K_M | 8 | 0 | 4.1 | 185.7 | 1739 | 0.72 | 0/8 | 0.0 | 3/8 | 아니오 | 7 |

## 8. 답변 원문 보기
`SHOW_QUESTION`에 질문 id를 넣으면 그 질문에 대한 모델별 답변을 나란히 보여 줍니다.
전체 답변은 `data/llm_compare/answers.md`에도 저장합니다.

In [8]:
SHOW_QUESTION = "hallucination_bait"   # QUESTIONS의 id 중 하나


def answer_block(row: dict[str, Any]) -> str:
    verdict = "검증 통과" if row["verify_passed"] else "검증 실패: " + ", ".join(row["unsupported"])
    return (f"### {row['model']}\n*{row['wall_seconds']}s · 인용 {row['citation_count']} · {verdict}*\n\n"
            + (row["answer"] or f"(오류: {row['error']})") + "\n")


sections = []
for item in QUESTIONS:
    group = [r for r in results if r["question_id"] == item["id"]]
    sections.append(f"## {item['id']}: {item['q']}\n\n" + "\n".join(answer_block(r) for r in group))
(OUT_DIR / "answers.md").write_text("\n".join(sections), encoding="utf-8")

question = next(item for item in QUESTIONS if item["id"] == SHOW_QUESTION)
display(Markdown(f"## {question['id']}: {question['q']}\n\n"
                 + "\n".join(answer_block(r) for r in results if r["question_id"] == SHOW_QUESTION)))

## hallucination_bait: 2025년 화성 이주민 지원 조례를 처음 통과시킨 의회와 예산 규모는?

### exaone3.5:7.8b
*4.45s · 인용 0 · 검증 통과*

제시된 근거들에 따르면, 2025년 화성 이주민 지원 조례에 대한 내용은 직접적으로 언급되어 있지 않습니다. 주어진 회의록 근거들은 주로 경기도의회, 울산광역시의회, 광주광역시의회, 그리고 일부 울산광역시의회 회의록의 내용을 포함하고 있으며, 이들 중 어느 곳에서도 화성 이주민 지원 조례에 대한 정보나 관련 예산 규모에 대한 언급을 찾을 수 없습니다.

따라서, 현재 제공된 정보로는 2025년 화성 이주민 지원 조례를 처음 통과시킨 의회와 그에 따른 예산 규모를 특정할 수 없습니다. [근거 부족]

### hf.co/mykor/A.X-4.0-Light-gguf:Q4_K_M
*6.76s · 인용 7 · 검증 통과*

2025년 화성 이주민 지원 조례를 처음 통과시킨 의회에 대한 정보는 주어진 검색 근거에 포함되어 있지 않습니다. 따라서 이에 대한 정확한 답변을 제공하기 어렵습니다.

검색된 내용 중 2025년 예산 관련 정보는 다음과 같습니다:

- **경기도의회**:
  - **2025년도 일반회계 예산안**: 총 34조 7,398억 원으로 138억 원 증액 [근거 6]
  - **2025년도 특별회계 예산안**: 총 3조 9,823억 원으로 2억 원 증액 [근거 6]
  - **주요 세출예산 증액 항목**: 경기청년 역량강화 지원 200억 원, 소상공인 지원 50억 원, 농수산물 할인쿠폰 지원 50억 원 등 [근거 6]

- **울산광역시의회**:
  - **산후조리비 지원**: 1억 7000만 원 증액 [근거 2]
  - **산모·신생아 건강관리 지원**: 5억 600만 원 증액 [근거 2]
  - **난임부부 시술비 지원**: 5억 3000만 원 증액 [근거 2]
  - **2023년 대비 예산 증가**: 예산 집행률이 높은 것으로 보아 예산 부족 문제 해결 [근거 2]

화성 이주민 지원 조례와 관련된 구체적인 예산 규모나 의회에 대한 정보는 제공된 검색 근거에서 찾을 수 없습니다. 추가적인 자료 확인이 필요합니다.

### hf.co/mradermacher/kanana-1.5-8b-instruct-2505-GGUF:Q4_K_M
*11.16s · 인용 6 · 검증 통과*

질문: 2025년 화성 이주민 지원 조례를 처음 통과시킨 의회와 예산 규모는?

[근거 1] 경기도의회 | 2024-12-27  
[근거 2] 울산광역시의회 | 2025-06-11  
[근거 3] 울산광역시의회 | 2025-06-16  
[근거 4] 광주광역시의회 | 2024-12-03  
[근거 5] 울산광역시의회 | 2025-06-11  
[근거 6] 경기도의회 | 2024-12-30  

**근거에 근거한 답변:**

질문에서 "2025년 화성 이주민 지원 조례"에 대한 직접적인 통과 사실이나 예산 규모에 대한 정보는 상기 제공된 근거(1~6)에 명시되어 있지 않습니다.  
각 근거는 주로 타 시·도(울산, 광주, 경기)의회에서 추경예산, 기금, 인구영향평가, 지원사업 등과 관련한 논의 및 예산 편성에 대해 다루고 있으나, "화성 이주민 지원 조례"에 대한 구체적 통과 시기와 예산 규모에 대한 언급은 없습니다.

**따라서, 해당 근거만으로는 2025년 화성 이주민 지원 조례의 최초 통과 의회와 예산 규모를 알 수 없습니다.**  
조례의 최초 통과 및 관련 예산에 대한 추가 자료(회의록, 조례안 제출 및 가결 현황 등)가 필요합니다.

**요약:**  
해당 근거에는 "2025년 화성 이주민 지원 조례"의 최초 통과 의회와 예산 규모에 대한 정보가 없습니다.  
(근거 부족)<|eot_id|>

### hf.co/DevQuasar/trillionlabs.Tri-7B-GGUF:Q4_K_M
*19.51s · 인용 11 · 검증 통과*

질문에 대한 답변은 다음과 같습니다.

**질문:** 2025년 화성 이주민 지원 조례를 처음 통과시킨 의회와 예산 규모는?

**답변:**
제공된 근거 자료 [근거 1], [근거 2], [근거 3], [근거 4], [근거 5], [근거 6]을 검토한 결과, 2025년 화성 이주민 지원 조례에 대한 직접적인 정보는 찾을 수 없었습니다. 이 자료들은 각각 경기도, 울산광역시, 광주광역시, 경기도의 예산 및 조례 관련 내용을 다루고 있으나, 화성 이주민 지원 조례에 대한 구체적인 정보는 포함되어 있지 않습니다.

따라서, **제공된 근거 자료만으로는 2025년 화성 이주민 지원 조례를 처음 통과시킨 의회와 예산 규모를 확인할 수 없습니다.**

**추가 정보:**
- 경기도의회 자료 ([근거 1])와 경기도의회 ([근거 6])는 2025년 경기도 예산 증액을 다루고 있으나, 화성 이주민 지원 조례에 대한 내용은 언급되지 않았습니다.
- 울산광역시의회 자료 ([근거 2], [근거 3])는 산후조리비 지원, 난임부부 시술비 지원 등 저출산 관련 예산 증액을 다루고 있으며, 화성 이주민 지원 조례와는 무관합니다.
- 광주광역시의회 자료 ([근거 4])는 광주광역시의 예산 증액을 설명하고 있으나, 화성 이주민 지원 조례와 관련 없습니다.

**결론:**
주어진 근거 자료만으로는 질문의 답변(2025년 화성 이주민 지원 조례를 처음 통과시킨 의회와 예산 규모)을 제공할 수 없습니다. 관련 조례나 예산에 대한 추가 정보나 다른 출처를 확인해야 할 것으로 보입니다.

### hf.co/rippertnt/HyperCLOVAX-SEED-Text-Instruct-1.5B-Q4_K_M-GGUF:Q4_K_M
*0.82s · 인용 2 · 검증 통과*

답변: 2025년 화성 이주민 지원 조례를 처음 통과시킨 의회는 경기도의회이며, 해당 조례의 예산 규모는 [근거 1] 138억 원 증액된 총 34조 7,398억 원입니다 [근거 2].

### hf.co/LGAI-EXAONE/EXAONE-4.0-1.2B-GGUF:Q4_K_M
*4.92s · 인용 0 · 검증 실패: 17000, 170000, 22.37, 34.7388, 347, 347.398, 3473, 46, 530, 560, 2024-12*

Hmm, the user is asking about the 2025 Mars colony support ordinance passed by the Gyeonggi Province Council and its budget size. Looking at the Korean sources provided, I see several pieces of information.

First, in Geumgongsi Council's December 2024 report (근거 1), they introduced an ordinance increasing medical reserve funds by 138 billion won. The total general account budget for 2025 was expanded to 34.73877 trillion won (347.398 billion won). This appears to be the core budget package.

The key detail comes from 근거 4 regarding Gwangju City Council's December 2024 report. They mentioned:
- Setoutput budget increased by 30.6% to 22.37 billion won
- Population impact assessment design service costs added 46 million won
- Economic alliance event costs for Honam region mega-city event costing 20 million won
- Also highlighted newborn health management support increasing annually

While there's no explicit mention of "화성 이주민 지원" in these reports, the broader context of special budget lines like specialaccounts shows Mars colony preparations are likely included. The 347 billion won general account budget stands as the primary indicator of scale for major space mission preparations.

The user seems to want precise legislative details, but the sources don't specify exact Mars colony parameters beyond the budget figures. I'll present what's clearly documented while noting the potential scope from special accounts mentioned in other reports.
</think>

2025년 화성 이주민 지원 조례와 관련된 주요 정보를 근거 자료를 바탕으로 정리하면 다음과 같습니다:

1. **조례 통과 배경 및 주요 내용**  
   - 경기도 일반의회(2024년 12월)에서 화성 이주민 지원을 위한 예산 증액을 포함한 세출예산안이 통과되었습니다.
   - 증액 규모: 의료급여 관련 예탁금(의료급여기금) 138억 원 증가, 청년 역량 강화 기회 지원(운전면허·자격증 취득 비용), 소상공인 지원, 농수산물 할인쿠폰 등이 반영되었습니다.

2. **2025년 경기도 예산안 규모**  
   - **일반회계 예산안**: 전년 대비 13.8% 증액(약 138억 원)을 반영해 총 **3473조 7387억 원**(34조 7,398억 원)으로 편성되었습니다.  
     - 세출예산 증액 주요 항목:
       * 청년 지원: 3,397억 원(운전면허/자격증 취득 지원 포함)
       * 소상공인 지원: 50억 원( handgun card 등)
       * 농수산물 지원: 50억 원
       * 기타 부대비(GH복합시설 임차료 등): 68억 원
   - 특별회계 예산안은 별도로 운영되며, 화성 관련 시설(소방안전·저상소방차 등) 지원 등이 포함됨을 확인할 수 있습니다.

3. **지원 확대 사항**  
   - 산후조리비· 산모·신생아 건강관리 예산이 지속적 증가하며(17,000만 원 → 170,000만 원+560만 원+530만 원), 저출산 대응 정책 일환으로 분석됐습니다.

※ 주의: 해당 자료에서는 공식적으로 "화성 이주민 지원 조례" 제목이 명시되지는 않았으나, 이주민 정착 지원을 포함한 포괄적 예산 증액이 초점으로 다루어졌습니다. 특별회계 활용 가능성도 고려해 세부 편성 내용이 추가될 수 있습니다.


## 9. 해석 시 유의점
- 질문 8개의 소규모 비교라 문항 하나의 차이가 크게 보입니다. 순위보다 **오류 유형**(지어낸 숫자, 인용 누락, 외국어 섞임)을 보세요.
- 검증은 숫자·날짜·의회명만 대조합니다. 근거와 다른 **내용**을 말해도 숫자가 없으면 통과할 수 있습니다.
- 응답 시간은 GPU·양자화(Q4_K_M)·답변 길이에 좌우됩니다. 같은 PC에서 같은 설정으로 잰 상대값으로만 비교하세요.
- 노트북의 채팅은 근거 외에 이슈 분석·예측·외부 데이터 요약도 함께 넣습니다. 이 비교는 **회의록 근거만** 넣은 조건입니다.
- 모델을 바꾸려면 제출 폴더 `.env`의 `COUNCIL_AGENT_MODEL`을 바꾸면 됩니다.